[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/blob/main/2.%20Go%20Beyond%20the%20Numbers%20Translate%20Data%20into%20Insights/trabajo_final_automatidata_eda.ipynb)


# **Automatidata Project: Exploratory Data Analysis**

**Course 2 — Go Beyond the Numbers: Translate Data into Insights**  
**Google Advanced Data Analytics Professional Certificate**

Este notebook desarrolla un análisis exploratorio de los viajes de taxis amarillos de Nueva York. Incluye:

- carga y revisión del dataset;
- limpieza y transformación de fechas;
- detección de valores atípicos;
- visualizaciones de distancias, montos, propinas y duración;
- análisis temporal por mes y día;
- conclusiones y recomendaciones para NYC TLC.


## **PACE: Plan**

### Objetivo

Comprender la estructura y calidad del dataset, detectar anomalías y construir visualizaciones que ayuden a explicar el comportamiento de los viajes.

### Tratamiento de valores atípicos

Se utiliza el método del rango intercuartílico (IQR), junto con boxplots e histogramas. Los valores extremos no se eliminan automáticamente:

- si corresponden a errores técnicos o de registro, deben corregirse o excluirse;
- si son observaciones reales, conviene analizarlas por separado;
- antes de modelar, se debe evaluar cuánto alteran las medias y relaciones entre variables.


In [ ]:
# Importar bibliotecas
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Configuración visual
sns.set_theme(style="whitegrid", context="notebook")
DEFAULT_COLOR = sns.color_palette("deep")[0]
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Bibliotecas importadas correctamente.")

## **1. Carga del dataset**

El notebook busca primero `2017_Yellow_Taxi_Trip_Data.csv` en el entorno local o en Google Colab. Si no lo encuentra, descarga automáticamente una copia pública desde GitHub. También valida que el archivo no esté vacío y que contenga las columnas esperadas.


In [ ]:
# Cargar el dataset de forma compatible con Google Colab, Jupyter y GitHub
from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen

DATA_FILENAME = "2017_Yellow_Taxi_Trip_Data.csv"

# Ruta exacta del archivo en el repositorio
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Abraham123w/"
    "Google-Advanced-Data-Analytics-Professional-Certificate/"
    "main/"
    "2.%20Go%20Beyond%20the%20Numbers%20Translate%20Data%20into%20Insights/"
    "2017_Yellow_Taxi_Trip_Data.csv"
)

# Posibles ubicaciones cuando el archivo fue subido manualmente
local_candidates = [
    Path(DATA_FILENAME),
    Path("/content") / DATA_FILENAME,
    Path("/mnt/data") / DATA_FILENAME,
]


def validate_dataframe(frame: pd.DataFrame, source: str) -> pd.DataFrame:
    """Verificar que el archivo cargado sea el dataset esperado."""

    if frame.empty:
        raise ValueError(
            f"El archivo obtenido desde {source} no contiene filas."
        )

    expected_columns = {
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "payment_type",
        "fare_amount",
        "tip_amount",
        "total_amount",
    }

    missing_columns = expected_columns.difference(frame.columns)

    if missing_columns:
        missing_text = ", ".join(sorted(missing_columns))
        raise ValueError(
            f"El archivo obtenido desde {source} no corresponde al dataset esperado. "
            f"Faltan estas columnas: {missing_text}"
        )

    return frame


def load_taxi_dataset():
    """Cargar el CSV desde Colab, Jupyter o GitHub."""

    errors = []

    # 1. Buscar primero una copia local
    for candidate in local_candidates:
        try:
            if candidate.exists():
                file_size = candidate.stat().st_size

                if file_size < 100:
                    raise ValueError(
                        f"El archivo local tiene solo {file_size} bytes."
                    )

                frame = pd.read_csv(candidate)
                frame = validate_dataframe(frame, str(candidate))

                return frame, str(candidate)

        except Exception as error:
            errors.append(f"{candidate}: {error}")

    # 2. Descargar automáticamente desde GitHub
    try:
        request = Request(
            DATA_URL,
            headers={"User-Agent": "Mozilla/5.0"},
        )

        with urlopen(request, timeout=45) as response:
            content = response.read()

        if len(content) < 100:
            raise ValueError(
                f"GitHub devolvió un archivo de solo {len(content)} bytes."
            )

        frame = pd.read_csv(BytesIO(content))
        frame = validate_dataframe(frame, DATA_URL)

        return frame, DATA_URL

    except Exception as error:
        errors.append(f"{DATA_URL}: {error}")

    error_details = "\n- ".join(errors)

    raise RuntimeError(
        "No fue posible cargar el dataset.\n"
        "Comprueba que el repositorio sea público, que la rama sea main y que "
        "el archivo se llame exactamente 2017_Yellow_Taxi_Trip_Data.csv.\n\n"
        f"Intentos realizados:\n- {error_details}"
    )


df, data_source = load_taxi_dataset()

print("Dataset cargado correctamente.")
print(f"Fuente: {data_source}")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Tamaño en memoria: {df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")

df.head()


## **PACE: Analyze**

### 2. Exploración inicial y calidad de los datos


In [ ]:
# Primeras observaciones
df.head(10)

In [ ]:
# Dimensiones y tamaño total
print(f"Tamaño del dataset: {df.size:,} valores")
print(f"Forma del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")

In [ ]:
# Tipos de datos y valores no nulos
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe().T

In [ ]:
# Revisión de valores nulos y duplicados
quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_count": df.isna().sum(),
    "null_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=False),
})

print(f"Filas duplicadas completas: {df.duplicated().sum():,}")
quality_summary

### Observaciones iniciales

- Las columnas de fecha se encuentran originalmente como texto y deben convertirse a `datetime`.
- `Unnamed: 0` parece ser un identificador o índice exportado.
- Se observan montos negativos y valores extremos que requieren validación.
- Algunos viajes tienen distancia o cantidad de pasajeros igual a cero.


In [ ]:
# Convertir columnas de fecha
datetime_columns = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]

for column in datetime_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

# Crear duración del viaje en minutos
df["trip_duration_minutes"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

print(df[datetime_columns + ["trip_duration_minutes"]].dtypes)
df[datetime_columns + ["trip_duration_minutes"]].head()

In [ ]:
# Resumen de valores potencialmente problemáticos
anomaly_summary = pd.Series({
    "pickup_datetime inválido": df["tpep_pickup_datetime"].isna().sum(),
    "dropoff_datetime inválido": df["tpep_dropoff_datetime"].isna().sum(),
    "distancia igual a 0": (df["trip_distance"] == 0).sum(),
    "pasajeros igual a 0": (df["passenger_count"] == 0).sum(),
    "fare_amount negativo": (df["fare_amount"] < 0).sum(),
    "total_amount negativo": (df["total_amount"] < 0).sum(),
    "duración <= 0 minutos": (df["trip_duration_minutes"] <= 0).sum(),
    "duración >= 120 minutos": (df["trip_duration_minutes"] >= 120).sum(),
})

anomaly_summary.to_frame("cantidad")

In [ ]:
# Función para detectar outliers mediante IQR
def iqr_outlier_summary(dataframe, columns):
    rows = []

    for column in columns:
        q1 = dataframe[column].quantile(0.25)
        q3 = dataframe[column].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = dataframe[(dataframe[column] < lower) | (dataframe[column] > upper)]

        rows.append({
            "variable": column,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "límite_inferior": lower,
            "límite_superior": upper,
            "cantidad_outliers": len(outliers),
            "porcentaje_outliers": len(outliers) / len(dataframe) * 100,
        })

    return pd.DataFrame(rows).set_index("variable")


outlier_columns = [
    "trip_distance",
    "total_amount",
    "tip_amount",
    "trip_duration_minutes",
]

iqr_outlier_summary(df, outlier_columns)

## **PACE: Construct**

### 3. Visualización de variables relevantes


In [ ]:
# Boxplot de distancia del viaje
plt.figure(figsize=(10, 3))
sns.boxplot(x=df["trip_distance"], fliersize=2, color=DEFAULT_COLOR)
plt.title("Trip Distance — Boxplot")
plt.xlabel("Distance (miles)")
plt.tight_layout()
plt.show()

In [ ]:
# Histograma de distancia del viaje
plt.figure(figsize=(11, 5))
sns.histplot(df["trip_distance"], bins=range(0, 36, 1), color=DEFAULT_COLOR)
plt.title("Trip Distance — Histogram")
plt.xlabel("Distance (miles)")
plt.ylabel("Ride count")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot del monto total
plt.figure(figsize=(10, 3))
sns.boxplot(x=df["total_amount"], fliersize=2, color=DEFAULT_COLOR)
plt.title("Total Amount — Boxplot")
plt.xlabel("Total amount (USD)")
plt.tight_layout()
plt.show()

In [ ]:
# Histograma del monto total, centrado en el rango principal
plt.figure(figsize=(12, 5))
sns.histplot(df["total_amount"], bins=range(-10, 105, 5), color=DEFAULT_COLOR)
plt.title("Total Amount — Histogram")
plt.xlabel("Total amount (USD)")
plt.ylabel("Ride count")
plt.xticks(range(-10, 105, 10))
plt.tight_layout()
plt.show()

### Propinas registradas en pagos con tarjeta

Las propinas en efectivo normalmente no quedan registradas electrónicamente. Por eso, para analizar `tip_amount`, se filtran los viajes con `payment_type == 1`.


In [ ]:
# Viajes pagados con tarjeta de crédito
credit_card_trips = df[df["payment_type"] == 1].copy()

print(f"Viajes con tarjeta: {len(credit_card_trips):,}")
print(f"Propina promedio: USD {credit_card_trips['tip_amount'].mean():.2f}")

In [ ]:
# Boxplot de propinas con tarjeta
plt.figure(figsize=(10, 3))
sns.boxplot(x=credit_card_trips["tip_amount"], fliersize=2, color=DEFAULT_COLOR)
plt.title("Tip Amount — Credit Card Payments")
plt.xlabel("Tip amount (USD)")
plt.tight_layout()
plt.show()

In [ ]:
# Histograma de propinas con tarjeta
plt.figure(figsize=(12, 5))
sns.histplot(credit_card_trips["tip_amount"], bins=range(0, 21, 1), color=DEFAULT_COLOR)
plt.title("Tip Amount — Credit Card Payments")
plt.xlabel("Tip amount (USD)")
plt.ylabel("Ride count")
plt.xticks(range(0, 21, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Propinas por proveedor
plt.figure(figsize=(12, 6))
sns.histplot(
    data=credit_card_trips,
    x="tip_amount",
    bins=range(0, 21, 1),
    hue="VendorID",
    multiple="dodge",
    shrink=0.8,
)
plt.title("Tip Amount by Vendor — Credit Card Payments")
plt.xlabel("Tip amount (USD)")
plt.ylabel("Ride count")
plt.xticks(range(0, 21, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Propinas generosas: más de USD 10
generous_tips = credit_card_trips[credit_card_trips["tip_amount"] > 10]

plt.figure(figsize=(12, 6))
sns.histplot(
    data=generous_tips,
    x="tip_amount",
    bins=range(10, 52, 2),
    hue="VendorID",
    multiple="dodge",
    shrink=0.8,
)
plt.title("Generous Tips by Vendor — Tips Above USD 10")
plt.xlabel("Tip amount (USD)")
plt.ylabel("Ride count")
plt.xticks(range(10, 52, 2))
plt.tight_layout()
plt.show()

### Propina promedio según cantidad de pasajeros


In [ ]:
# Frecuencia de viajes según cantidad de pasajeros
df["passenger_count"].value_counts().sort_index()

In [ ]:
# Propina promedio según cantidad de pasajeros
mean_tips_by_passenger = (
    df.groupby("passenger_count", as_index=False)["tip_amount"]
      .mean()
      .rename(columns={"tip_amount": "mean_tip_amount"})
)

mean_tips_by_passenger

In [ ]:
# Excluir passenger_count = 0 del gráfico comparativo
plot_data = mean_tips_by_passenger[mean_tips_by_passenger["passenger_count"] > 0]

plt.figure(figsize=(11, 6))
sns.barplot(
    data=plot_data,
    x="passenger_count",
    y="mean_tip_amount",
    color=DEFAULT_COLOR,
)
plt.axhline(
    df["tip_amount"].mean(),
    linestyle="--",
    linewidth=1.5,
    label=f"Global mean: USD {df['tip_amount'].mean():.2f}",
)
plt.title("Mean Tip Amount by Passenger Count")
plt.xlabel("Passenger count")
plt.ylabel("Mean tip (USD)")
plt.legend()
plt.tight_layout()
plt.show()

### Análisis temporal por mes y día de la semana


In [ ]:
# Crear columnas temporales
df["month"] = df["tpep_pickup_datetime"].dt.month_name()
df["day"] = df["tpep_pickup_datetime"].dt.day_name()
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["quarter"] = df["tpep_pickup_datetime"].dt.to_period("Q").astype(str)

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]

df[["month", "day", "pickup_hour", "quarter"]].head()

In [ ]:
# Cantidad total de viajes por mes
monthly_rides = df["month"].value_counts().reindex(month_order)

plt.figure(figsize=(13, 6))
sns.barplot(x=monthly_rides.index, y=monthly_rides.values, color=DEFAULT_COLOR)
plt.title("Ride Count by Month")
plt.xlabel("Month")
plt.ylabel("Ride count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

monthly_rides.to_frame("ride_count")

In [ ]:
# Cantidad total de viajes por día de la semana
daily_rides = df["day"].value_counts().reindex(day_order)

plt.figure(figsize=(11, 6))
sns.barplot(x=daily_rides.index, y=daily_rides.values, color=DEFAULT_COLOR)
plt.title("Ride Count by Day of Week")
plt.xlabel("Day")
plt.ylabel("Ride count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

daily_rides.to_frame("ride_count")

In [ ]:
# Ingreso total por día de la semana
revenue_by_day = (
    df.groupby("day")["total_amount"]
      .sum()
      .reindex(day_order)
)

plt.figure(figsize=(11, 6))
sns.barplot(x=revenue_by_day.index, y=revenue_by_day.values, color=DEFAULT_COLOR)
plt.title("Total Revenue by Day of Week")
plt.xlabel("Day")
plt.ylabel("Total revenue (USD)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

revenue_by_day.to_frame("total_revenue")

In [ ]:
# Ingreso total por mes
revenue_by_month = (
    df.groupby("month")["total_amount"]
      .sum()
      .reindex(month_order)
)

plt.figure(figsize=(13, 6))
sns.barplot(x=revenue_by_month.index, y=revenue_by_month.values, color=DEFAULT_COLOR)
plt.title("Total Revenue by Month")
plt.xlabel("Month")
plt.ylabel("Total revenue (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

revenue_by_month.to_frame("total_revenue")

In [ ]:
# Viajes por trimestre
rides_by_quarter = df["quarter"].value_counts().sort_index()

plt.figure(figsize=(9, 5))
sns.lineplot(
    x=rides_by_quarter.index,
    y=rides_by_quarter.values,
    marker="o",
    color=DEFAULT_COLOR,
)
plt.title("Ride Count by Quarter")
plt.xlabel("Quarter")
plt.ylabel("Ride count")
plt.tight_layout()
plt.show()

rides_by_quarter.to_frame("ride_count")

In [ ]:
# Demanda según hora de recogida
rides_by_hour = df["pickup_hour"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
sns.lineplot(
    x=rides_by_hour.index,
    y=rides_by_hour.values,
    marker="o",
    color=DEFAULT_COLOR,
)
plt.title("Ride Count by Pickup Hour")
plt.xlabel("Hour of day")
plt.ylabel("Ride count")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

### Relación entre distancia y monto total


In [ ]:
# Filtrar valores razonables para visualizar la relación principal
scatter_data = df[
    (df["trip_distance"] > 0)
    & (df["trip_distance"] <= 35)
    & (df["total_amount"] > 0)
    & (df["total_amount"] <= 200)
]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=scatter_data,
    x="trip_distance",
    y="total_amount",
    alpha=0.35,
    s=25,
    color=DEFAULT_COLOR,
)
plt.title("Trip Distance vs. Total Amount")
plt.xlabel("Trip distance (miles)")
plt.ylabel("Total amount (USD)")
plt.tight_layout()
plt.show()

correlation = scatter_data[["trip_distance", "total_amount"]].corr().iloc[0, 1]
print(f"Correlación de Pearson en el rango visualizado: {correlation:.3f}")

### Distancia promedio y cantidad de viajes por destino


In [ ]:
# Cantidad de ubicaciones de destino únicas
num_dropoff_locations = df["DOLocationID"].nunique()
print(f"Ubicaciones de destino únicas: {num_dropoff_locations}")

In [ ]:
# Distancia promedio por ubicación de destino
distance_by_dropoff = (
    df.groupby("DOLocationID")["trip_distance"]
      .mean()
      .sort_values()
)

plt.figure(figsize=(18, 7))
sns.barplot(
    x=distance_by_dropoff.index.astype(str),
    y=distance_by_dropoff.values,
    order=distance_by_dropoff.index.astype(str),
    color=DEFAULT_COLOR,
)
plt.title("Mean Trip Distance by Drop-off Location")
plt.xlabel("Drop-off Location ID")
plt.ylabel("Mean distance (miles)")
plt.xticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Verificar si los IDs de destino son consecutivos
unique_ids = np.sort(df["DOLocationID"].unique())
ideal_range = np.arange(unique_ids.min(), unique_ids.max() + 1)
missing_ids = np.setdiff1d(ideal_range, unique_ids)

print(f"Rango de IDs: {unique_ids.min()}–{unique_ids.max()}")
print(f"Cantidad de IDs observados: {len(unique_ids)}")
print(f"Cantidad de IDs faltantes en la secuencia: {len(missing_ids)}")
print(f"Primeros IDs faltantes: {missing_ids[:20]}")

In [ ]:
# Cantidad de viajes por ubicación de destino
dropoff_counts = df["DOLocationID"].value_counts().sort_values()

plt.figure(figsize=(18, 7))
sns.barplot(
    x=dropoff_counts.index.astype(str),
    y=dropoff_counts.values,
    order=dropoff_counts.index.astype(str),
    color=DEFAULT_COLOR,
)
plt.title("Rides by Drop-off Location — Ordered by Popularity")
plt.xlabel("Drop-off Location ID")
plt.ylabel("Ride count")
plt.xticks([])
plt.tight_layout()
plt.show()

### Duración de los viajes

La duración se calcula a partir de las fechas de recogida y llegada. Para mostrar la distribución principal se utiliza un subconjunto entre 1 y 120 minutos, sin eliminar los registros del dataframe original.


In [ ]:
# Estadísticas de duración antes de aplicar filtros visuales
df["trip_duration_minutes"].describe()

In [ ]:
# Filtrado para una visualización interpretable
duration_clean = df[
    (df["trip_duration_minutes"] > 1)
    & (df["trip_duration_minutes"] < 120)
].copy()

print(f"Viajes incluidos en la visualización: {len(duration_clean):,}")
print(f"Duración promedio: {duration_clean['trip_duration_minutes'].mean():.2f} minutos")

In [ ]:
# Boxplot de duración
plt.figure(figsize=(11, 3))
sns.boxplot(x=duration_clean["trip_duration_minutes"], fliersize=2, color=DEFAULT_COLOR)
plt.title("Trip Duration — Boxplot")
plt.xlabel("Duration (minutes)")
plt.tight_layout()
plt.show()

In [ ]:
# Histograma accesible de duración
plt.figure(figsize=(12, 6))
sns.histplot(
    data=duration_clean,
    x="trip_duration_minutes",
    bins=30,
    kde=True,
    color=DEFAULT_COLOR,
)
mean_duration = duration_clean["trip_duration_minutes"].mean()
plt.axvline(
    mean_duration,
    linestyle="--",
    linewidth=2,
    label=f"Mean: {mean_duration:.2f} minutes",
)
plt.title("Distribution of Trip Duration")
plt.xlabel("Duration (minutes)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## **PACE: Execute**

### Resultados y evaluación

El análisis exploratorio permitió identificar patrones de demanda, comportamiento de las propinas, variaciones temporales y problemas de calidad que deben atenderse antes de modelar.

Los principales riesgos de calidad son:

- montos negativos;
- viajes con distancia cero;
- duraciones negativas o excesivamente largas;
- valores extremos en montos y propinas;
- una columna de índice que probablemente no aporta valor analítico.

Preguntas adicionales para NYC TLC:

1. ¿Qué horas concentran la mayor demanda y el mayor ingreso?
2. ¿Qué zonas de origen y destino generan los viajes más rentables?
3. ¿Cómo cambia la propina según hora, día, distancia y proveedor?
4. ¿Qué registros extremos corresponden a errores, reembolsos o viajes válidos?
5. ¿Cuál es el ingreso promedio por milla y por minuto?

Las visualizaciones deben mantener etiquetas claras, contraste suficiente y evitar depender exclusivamente del color.


In [ ]:
# Resumen ejecutivo calculado desde el dataset
executive_summary = pd.Series({
    "viajes": len(df),
    "variables originales": 18,
    "valores nulos originales": int(df.iloc[:, :18].isna().sum().sum()),
    "distancia promedio (millas)": df["trip_distance"].mean(),
    "monto total promedio (USD)": df["total_amount"].mean(),
    "propina promedio tarjeta (USD)": credit_card_trips["tip_amount"].mean(),
    "duración mediana (minutos)": df["trip_duration_minutes"].median(),
    "mes con más viajes": monthly_rides.idxmax(),
    "día con más viajes": daily_rides.idxmax(),
    "día con mayor ingreso": revenue_by_day.idxmax(),
})

executive_summary.to_frame("resultado")

## **Conclusión**

El EDA es fundamental porque permite comprender la estructura y distribución de los datos, detectar anomalías y evaluar la calidad antes de desarrollar cualquier modelo predictivo.

En este proyecto, las visualizaciones mostraron que la mayoría de los viajes tienen distancias, duraciones y montos relativamente bajos, mientras que existe una cola derecha con observaciones extremas. También se identificaron diferencias temporales en la demanda y el ingreso.

Para una siguiente etapa, conviene:

- validar los registros negativos y extremos;
- crear variables de duración, hora, día y mes;
- incorporar información geográfica;
- construir un modelo base que use `trip_distance` y variables temporales para predecir `total_amount`;
- comparar el desempeño del modelo con y sin observaciones atípicas.


## **Exportación opcional**

Ejecuta la siguiente celda únicamente cuando quieras guardar una versión enriquecida del dataset.


In [ ]:
# Descomenta para guardar el dataset enriquecido
# output_path = "2017_Yellow_Taxi_Trip_Data_enriched.csv"
# df.to_csv(output_path, index=False)
# print(f"Archivo guardado: {output_path}")